# 03 LSTM Sequence Model

This notebook focuses on the sequential model family. The LSTM differs from the tabular models because it keeps each lookback input as a time-by-feature sequence and also uses station identity through an embedding layer.

The purpose of this notebook is to make the sequence logic understandable: why windows must not cross stream boundaries, how the final timestamp in a window receives the prediction target, and how the LSTM results compare to the rest of the high-recall detection pipeline.

This notebook is useful for the report because it explains why the LSTM is included even if the main contribution is the XAI alarm-triage layer. The LSTM gives a different model family for checking whether temporal evidence is used differently from flattened tabular models.


## Load Data and LSTM Results

This cell loads the consolidated dataset and filters the final model outputs to the LSTM.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / "nita_xai").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "nita_xai").exists())

sys.path.insert(0, str(PROJECT_ROOT))
from nita_xai import config

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

training = pd.read_csv(config.PROCESSED_DIR / "training_dataset.csv")
metrics = pd.read_csv(config.FINAL_MODEL_METRICS_PATH)
comparison = pd.read_csv(config.FINAL_MODEL_COMPARISON_PATH)
thesis_metrics = pd.read_csv(config.THESIS_STYLE_MODEL_METRICS_PATH)
predictions = pd.read_csv(config.FINAL_PREDICTIONS_DIR / "all_model_predictions.csv")

lstm_metrics = metrics[metrics["model_name"] == "lstm"]
lstm_predictions = predictions[predictions["model"] == "lstm"]
print(f"LSTM prediction rows: {len(lstm_predictions):,}")


LSTM prediction rows: 23,100


these are saved predictions from the final trained LSTM folds. The notebook is not retraining the sequence model.


## Sequence Window Logic

The LSTM uses lookback windows. A window must stay within one stream, because crossing from one incident stream into another would create artificial temporal context and data leakage.


In [2]:
sequence_notes = pd.DataFrame({
    "setting": ["lookback", "features", "station context", "boundary rule"],
    "value": [
        config.LOOKBACK,
        ", ".join(config.FEATURE_COLUMNS),
        "station-ID embedding",
        "windows stay inside one stream/source file",
    ],
})
sequence_notes


,setting,value
0,lookback,12
1,features,"speed_smoothed, occ_smoothed"
2,station context,station-ID embedding
3,boundary rule,windows stay inside one stream/source file


the final timestamp in each lookback window receives the prediction label. This is why row identity must be preserved carefully through training, prediction, XAI, and triage.


## LSTM Fold Metrics

These fold-level metrics show how the sequence model behaved across the five held-out groups.


In [3]:
lstm_metrics.sort_values("fold")


,model_name,fold,threshold,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,train_windows,calibration_windows,test_windows,model_path,score_path,run_mode,threshold_strategy,epochs_trained
0,lstm,1,0.071721,0.549865,0.608955,0.577904,0.886130,0.582683,204,167,131,4118,320880,80640,4620,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_1\model.pt,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_1\lstm_fold_1_scores.csv,final,baseline_calibration_quantile,29.0
4,lstm,2,0.071195,0.501027,0.668493,0.572770,0.906626,0.600582,244,243,121,4012,320880,80640,4620,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_2\model.pt,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_2\lstm_fold_2_scores.csv,final,baseline_calibration_quantile,21.0
8,lstm,3,0.070524,0.532558,0.585678,0.557856,0.903606,0.599001,229,201,162,4028,320880,80640,4620,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_3\model.pt,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_3\lstm_fold_3_scores.csv,final,baseline_calibration_quantile,30.0
12,lstm,4,0.069529,0.503080,0.698006,0.584726,0.903267,0.637311,245,242,106,4027,320880,80640,4620,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_4\model.pt,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_4\lstm_fold_4_scores.csv,final,baseline_calibration_quantile,18.0
16,lstm,5,0.069746,0.557447,0.685864,0.615023,0.915119,0.621720,262,208,120,4030,320880,80640,4620,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_5\model.pt,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\nita_xai\models\final\lstm\fold_5\lstm_fold_5_scores.csv,final,baseline_calibration_quantile,30.0


compare recall and false positives across folds. The LSTM may detect temporal behavior differently from tabular models, which is why its explanations are compared separately.


## LSTM Alarm Behavior

This summarizes LSTM timestamp predictions by case type.


In [4]:
lstm_case_summary = (
    lstm_predictions.groupby("case_type")
    .size()
    .reset_index(name="timestamps")
    .sort_values("timestamps", ascending=False)
)
lstm_case_summary


,case_type,timestamps
2,true_negative,20215
3,true_positive,1184
0,false_positive,1061
1,missed_incident,640


false positives are expected in a high-recall setting. The later triage notebook groups these timestamps into episodes so the operator view is not overloaded.


## LSTM Limitations for XAI

SHAP and LIME are easier to apply to tabular models. For the LSTM, the explanation method must respect the sequence shape. In this project, the script saves sequence-style attribution tables where the evidence can be interpreted across feature and time-step dimensions.


## LSTM Incident-Level Detection View

The timestamp-level recall for LSTM is stricter than thesis Detection Rate. This cell shows the incident-level view, where an incident counts as detected if at least one alarm occurs inside the labeled incident period.


In [5]:
thesis_metrics[thesis_metrics["model"] == "lstm"]


,model,detection_rate_mean,false_alarm_rate_mean,mean_time_to_detection_minutes,median_time_to_detection_minutes,mean_alarm_coverage_inside_incident,incident_count_total,detected_incidents_total,missed_incidents_total,false_alarm_timestamps_total,normal_timestamps_total,metric_note
0,lstm,0.994286,0.049876,0.543697,0.0,0.681167,175,174,1,1061,21276,Thesis-style incident-level DR with timestamp-level FAR; MTTD is measured from incident start to first alarm inside the labeled incident period.


**Interpretation:** this is the number to use when explaining why the LSTM is still broadly consistent with the thesis-style incident-detection framing, even though our timestamp-level recall is lower.
